# W02 — Quantum Gates 101
**Q-BITS · BITS Pilani Dubai Campus · Track 0: Foundations**

---

Quantum gates are the operations we apply to qubits. Every gate is a **unitary matrix** — meaning it is reversible and preserves the normalisation of the state vector.

**Prerequisites:** W01 — you should be comfortable with the state vector $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ and the Bloch sphere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector

print('Imports successful.')

---

## 1. Why gates must be unitary

A gate $U$ acting on state $|\psi\rangle$ produces $U|\psi\rangle$. For the output to remain a valid quantum state, normalisation must be preserved:

$$\langle\psi|U^\dagger U|\psi\rangle = \langle\psi|\psi\rangle = 1$$

This requires $U^\dagger U = I$ — the definition of a **unitary matrix**.

A useful consequence: every quantum gate is reversible. Its inverse is $U^\dagger$ (conjugate transpose). This is fundamentally different from classical logic — you cannot build an irreversible AND gate in quantum computing.

In [ ]:
# Verify unitarity: U†U = I
# Using the Hadamard gate as an example

H = np.array([[1, 1],
              [1, -1]]) / np.sqrt(2)

product = H.conj().T @ H
print('H†H =\n', np.round(product, 4))
print('\nIs unitary:', np.allclose(product, np.eye(2)))

---

## 2. Single-qubit gates

### X gate (quantum NOT)

Flips |0⟩ ↔ |1⟩. The quantum analogue of a classical NOT gate.

$$X = \begin{pmatrix}0 & 1\\1 & 0\end{pmatrix}, \quad X|0\rangle = |1\rangle, \quad X|1\rangle = |0\rangle$$

On the Bloch sphere: 180° rotation around the X axis.

In [ ]:
# X gate: flip |0> to |1>
qc = QuantumCircuit(1)
qc.x(0)
sv = Statevector(qc)

print(qc.draw('text'))
print('State after X|0>:', sv)
plot_bloch_multivector(sv)

### H gate (Hadamard)

Creates equal superposition from a basis state. The most important single-qubit gate.

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1\\1 & -1\end{pmatrix}$$

$$H|0\rangle = |+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad H|1\rangle = |-\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

On the Bloch sphere: 180° rotation around the X+Z axis (maps Z to X and X to Z).

Note: $H^2 = I$ — applying H twice returns to the original state.

In [ ]:
# H gate on |0> and |1>
for init, label in [('0', '|0>'), ('1', '|1>')]:
    qc = QuantumCircuit(1)
    if init == '1':
        qc.x(0)
    qc.h(0)
    sv = Statevector(qc)
    print(f'H{label} = {np.round(sv.data, 4)}')

# Verify H² = I
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)
sv = Statevector(qc)
print(f'\nH²|0> = {np.round(sv.data, 4)}  ← should be [1, 0]')

### Z, S, T gates

These gates introduce **phase** — they shift the relative phase between |0⟩ and |1⟩ without changing measurement probabilities in the Z basis.

$$Z = \begin{pmatrix}1 & 0\\0 & -1\end{pmatrix}, \quad S = \begin{pmatrix}1 & 0\\0 & i\end{pmatrix}, \quad T = \begin{pmatrix}1 & 0\\0 & e^{i\pi/4}\end{pmatrix}$$

| Gate | Phase shift | Bloch sphere rotation |
|---|---|---|
| Z | 180° | π around Z axis |
| S | 90° | π/2 around Z axis |
| T | 45° | π/4 around Z axis |

Phase gates have no classical analogue — they matter when combined with H gates.

In [ ]:
# Compare Z, S, T on |+> state — phase differences become visible
fig, axes = plt.subplots(1, 4, figsize=(14, 4),
                          subplot_kw={'projection': '3d'})

gates = [
    ('|+⟩', lambda qc: None),
    ('Z|+⟩', lambda qc: qc.z(0)),
    ('S|+⟩', lambda qc: qc.s(0)),
    ('T|+⟩', lambda qc: qc.t(0)),
]

for ax, (label, apply_gate) in zip(axes, gates):
    qc = QuantumCircuit(1)
    qc.h(0)
    apply_gate(qc)
    sv = Statevector(qc)
    plot_bloch_multivector(sv, ax=ax)
    ax.set_title(label, fontsize=12)

plt.tight_layout()
plt.show()

---

## 3. The CNOT gate (CX)

CNOT is a **two-qubit gate** — the most important entangling gate. It has a **control** qubit and a **target** qubit:

- If control = |0⟩: target is unchanged
- If control = |1⟩: target is flipped (X applied)

$$\text{CNOT}|00\rangle = |00\rangle, \quad \text{CNOT}|01\rangle = |01\rangle$$
$$\text{CNOT}|10\rangle = |11\rangle, \quad \text{CNOT}|11\rangle = |10\rangle$$

As a 4×4 matrix:
$$\text{CNOT} = \begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{pmatrix}$$

In [ ]:
# CNOT truth table
print('CNOT truth table:')
print(f'{"Input":>10} → {"Output"}')

for ctrl, tgt in [(0,0),(0,1),(1,0),(1,1)]:
    qc = QuantumCircuit(2)
    if ctrl: qc.x(1)  # set control qubit
    if tgt:  qc.x(0)  # set target qubit
    qc.cx(1, 0)        # CNOT: control=q1, target=q0
    sv = Statevector(qc)
    out = sv.data
    result = int(np.argmax(np.abs(out)))
    out_ctrl = result >> 1
    out_tgt  = result & 1
    print(f'  |{ctrl}{tgt}> → |{out_ctrl}{out_tgt}>')

---

## 4. Circuit composition

Gates compose left-to-right in circuit diagrams, but right-to-left in matrix multiplication:

$$|\psi_{\text{out}}\rangle = U_3 U_2 U_1 |\psi_{\text{in}}\rangle$$

Order matters — quantum gates generally do **not** commute: $HX \neq XH$.

In [ ]:
# HX vs XH — order matters
qc_hx = QuantumCircuit(1)
qc_hx.x(0)
qc_hx.h(0)

qc_xh = QuantumCircuit(1)
qc_xh.h(0)
qc_xh.x(0)

sv_hx = Statevector(qc_hx)
sv_xh = Statevector(qc_xh)

print('HX|0> =', np.round(sv_hx.data, 4))  # |->
print('XH|0> =', np.round(sv_xh.data, 4))  # |+> then flipped
print('Are they equal?', np.allclose(sv_hx.data, sv_xh.data))

---

## 5. Exercises

**Exercise 1.** Apply X then H to |0⟩. What state do you get? Verify with the statevector.

**Exercise 2.** Show that $H^2 = I$ by computing $H \cdot H$ as numpy matrices.

**Exercise 3.** What does $Z|+\rangle$ produce? Compute it and plot it on the Bloch sphere. Why does the measurement probability in the Z basis not change?

**Exercise 4.** Build a circuit that flips the target qubit of a CNOT only when the control is |1⟩. Verify all four input combinations.

In [ ]:
# Your answers here


---

## Summary

| Gate | Matrix | Effect |
|---|---|---|
| X | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Flips \|0⟩↔\|1⟩ |
| H | $\frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | Creates superposition |
| Z | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Phase flip |
| S | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | 90° phase |
| T | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | 45° phase |
| CNOT | 4×4 | Flips target if control = \|1⟩ |

**Next workshop:** [W03 — Your First Circuit](../W03/notebook.ipynb)

---
*Q-BITS · BITS Pilani Dubai Campus*